# 第14周：调试优化、音效关卡、最终发布（Day 92-98）

这是一个可以自动检测练习题的 Jupyter Notebook，配合《第14周详细课程》使用。

**这个notebook每一天都包含两部分：📖 知识点讲解（复习本周课程的讲解要点和实际场景）+ ✏️ 练习题与🔍自动检测。**

**怎么用这个notebook：**
1. 从上到下按顺序运行每一个cell（选中cell后按 `Shift + Enter`）
2. 看到 `✏️ 你的代码` 的cell，把 `pass` 删掉，写上你的代码
3. 写完之后，运行下面紧跟着的 `🔍 检测` cell，看看是不是显示 ✅（通过）
4. 如果显示 ❌，看看提示信息，回去改一改代码再重新运行这两个cell（先改代码的cell，再改检测的cell）

**一个小提醒：** 这是"期末大项目"编码部分的最后一周了！这周不学新的语法知识点，而是把上周做好的"完整游戏框架"打磨得更正式：加上关卡表驱动的难度曲线、暂停功能、音效钩子、健壮性检查。Checkpoint会把这些全部整合进第13周的游戏类里，做出一个真正可以演示给别人看的"最终完成版"。完成这周之后，建议花1-2天做"发布会"——演示游戏、写一份游戏说明书，为这100天的学习画上一个圆满的句号！

这个notebook不会给你答案，只会告诉你做得对不对，加油，这是最后一周了！


## 📑 本周内容导航

每天的内容都是先有 **📖 知识点讲解**，再有 **✏️ 练习 + 🔍 检测**，请按顺序学习：

- Day 92：多关卡设计——用一张关卡表驱动难度
- Day 93：暂停功能——给游戏加一个暂停/继续状态
- Day 94：音效完善——给关键事件配上声音反馈
- Day 95：健壮性与边界情况——让程序不容易被意外输入搞崩溃
- Day 96：最终整合——把关卡、暂停、音效钩子都装进游戏类里
- Day 97：代码整理与游戏说明书
- Day 98：第14周 Checkpoint（期末大项目最终完成版）


In [ ]:
# 🔧 工具cell：检测练习题用的工具函数和pygame初始化，不需要修改，直接运行一次就好
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # 让pygame可以在没有真实显示器的环境里运行（这个notebook专用，自己电脑上写游戏不需要这一行）
os.environ["SDL_AUDIODRIVER"] = "dummy"
import pygame
import struct
import random
pygame.init()
pygame.mixer.init()
pygame.font.init()

def test_check(fn, desc):
    """fn 是一个不带参数的函数（通常是 lambda），运行它并根据结果打印 ✅ 或 ❌"""
    try:
        result = fn()
    except Exception as e:
        print(f"❌ {desc} —— 运行出错：{type(e).__name__}: {e}")
        return False
    ok = bool(result)
    if ok:
        print(f"✅ {desc}")
    else:
        print(f"❌ {desc}")
    return ok

# 这周需要用到第13周已经写过的两个函数，这里先直接提供，不需要重新写一遍
def is_point_in_rect(point, rect):
    return pygame.Rect(rect).collidepoint(point)

def get_clicked_button(events, button_rect):
    for e in events:
        if e.type == pygame.MOUSEBUTTONDOWN:
            if is_point_in_rect(e.pos, button_rect):
                return True
    return False

print("工具函数准备好了，可以开始做练习题了！")


## Day 92：多关卡设计——用一张关卡表驱动难度

**学习目标**：理解怎么用一份数据（而不是写一堆if/elif）来管理多个关卡的难度设置

### 📖 知识点讲解

**讲解要点**

- **前面几周的"难度递增"是用公式算出来的，但正式的游戏往往希望"关卡"是被精心设计、可以随时调整的**——最好不要把这些数字写死在很多个if里面，而是统一放进一张"关卡表"（一个列表，每一项是一个字典，记录"第几关、速度多少、过关需要多少分"）。想调整难度曲线时，只需要改这张表的数字，不需要去改代码逻辑。
  **实际场景**：很多游戏的关卡设计师并不直接写代码，而是在一张Excel表格里填"第几关、敌人数量、通关条件"这些数字，程序只负责"读表、按表执行"。

- **根据"当前在第几关"去关卡表里查找对应的配置，是一种很常见的"查表"操作**：正常情况下能在表里找到对应的那一项；但如果关卡数已经超过了表里设计的最大关卡，比较稳妥的做法是"返回最后一关的配置"，让游戏在"表外"也能继续运行，而不是直接报错崩溃。
  **实际场景**：很多游戏的"无尽模式"在设计师准备的关卡用完之后，会固定用最后一关（最高难度）的参数无限循环下去，而不是在玩家打穿关卡表后直接崩溃退出。


### ✏️ 练习1：`get_level_config(level, level_table)`

接收当前关卡号`level`和关卡表`level_table`（一个列表，每项是像`{"level": 1, "speed": 20, "score_to_pass": 30}`这样的字典）。如果`level_table`是空列表，`return None`。否则遍历`level_table`，找到`item["level"] == level`的那一项并`return`它；如果遍历完都没找到（说明关卡号超出了表的范围），`return level_table[-1]`（最后一项，当作"表外"时的默认难度）。


In [ ]:
# ✏️ 你的代码
def get_level_config(level, level_table):
    pass


In [ ]:
# 🔍 检测
results_92_1 = []
_table92 = [
    {"level": 1, "speed": 20, "score_to_pass": 30},
    {"level": 2, "speed": 30, "score_to_pass": 60},
    {"level": 3, "speed": 45, "score_to_pass": 9999},
]
results_92_1.append(test_check(lambda: get_level_config(1, _table92) == _table92[0], "根据关卡号找到对应配置"))
results_92_1.append(test_check(lambda: get_level_config(2, _table92) == _table92[1], "第二关配置正确"))
results_92_1.append(test_check(lambda: get_level_config(99, _table92) == _table92[-1], "超出关卡表范围时返回最后一关配置"))
results_92_1.append(test_check(lambda: get_level_config(1, []) is None, "关卡表为空时返回None"))
print(f"\n通过 {sum(results_92_1)}/{len(results_92_1)} 项")


### ✏️ 挑战：`check_level_complete(score, level_config)`

接收当前分数`score`和一关的配置`level_config`（里面有`"score_to_pass"`这个键）。判断`score`是否大于等于`level_config["score_to_pass"]`，`return`这个判断结果（`True`或`False`）。


In [ ]:
# ✏️ 你的代码
def check_level_complete(score, level_config):
    pass


In [ ]:
# 🔍 检测
results_92_2 = []
results_92_2.append(test_check(lambda: check_level_complete(35, _table92[0]) == True, "分数达到通关线时返回True"))
results_92_2.append(test_check(lambda: check_level_complete(20, _table92[0]) == False, "分数没达到通关线时返回False"))
print(f"\n通过 {sum(results_92_2)}/{len(results_92_2)} 项")


## Day 93：暂停功能——给游戏加一个暂停/继续状态

**学习目标**：掌握怎么在已有的状态机里，新增一个暂停状态，并用一个按键在玩/暂停之间切换

### 📖 知识点讲解

**讲解要点**

- **"暂停"本质上也是游戏的一种状态**（跟`menu`、`playing`、`game_over`是同一类东西），只是比较特殊：暂停的时候画面定住不动，但不是"结束"，按一下暂停键还能继续刚才的进度。所以可以在`playing`和一个新的`paused`状态之间来回切换，不需要新建一整套复杂的逻辑。
  **实际场景**：几乎所有视频播放器、游戏机都有"暂停"按钮——按一下暂停，画面停在当前位置；再按一下，从刚才的位置继续，而不是从头开始。

- **判断"该不该切换暂停状态"，需要先看当前按的是哪个键，再看当前状态允不允许被暂停**——比如在`menu`界面按"暂停键"应该没有任何效果，只有在`playing`或`paused`状态下，这个按键才有意义。
  **实际场景**：遥控器上的"暂停"按钮，在电视关机的时候按下去是没有反应的——只有"正在播放"的时候，暂停才是一个有意义的操作。


### ✏️ 练习1：`toggle_pause(state)`

接收当前状态`state`。如果`state == "playing"`，`return "paused"`；如果`state == "paused"`，`return "playing"`；其他情况（比如`"menu"`、`"game_over"`）直接`return state`（保持不变）。


In [ ]:
# ✏️ 你的代码
def toggle_pause(state):
    pass


In [ ]:
# 🔍 检测
results_93_1 = []
results_93_1.append(test_check(lambda: toggle_pause("playing") == "paused", "playing切换到paused"))
results_93_1.append(test_check(lambda: toggle_pause("paused") == "playing", "paused切换回playing"))
results_93_1.append(test_check(lambda: toggle_pause("menu") == "menu", "menu状态不受暂停键影响"))
print(f"\n通过 {sum(results_93_1)}/{len(results_93_1)} 项")


### ✏️ 挑战：`handle_pause_key(state, events, pause_key)`

接收当前状态`state`、事件列表`events`、暂停键的键值`pause_key`（比如`pygame.K_p`）。遍历`events`，如果找到一个`pygame.KEYDOWN`类型的事件，并且它的`.key`等于`pause_key`，就调用Day93练习1的`toggle_pause(state)`并把结果`return`出去；如果遍历完都没找到这样的事件，`return state`（保持不变）。


In [ ]:
# ✏️ 你的代码
def handle_pause_key(state, events, pause_key):
    pass


In [ ]:
# 🔍 检测
results_93_2 = []
_events_p93 = [pygame.event.Event(pygame.KEYDOWN, key=pygame.K_p)]
_events_other93 = [pygame.event.Event(pygame.KEYDOWN, key=pygame.K_a)]
results_93_2.append(test_check(lambda: handle_pause_key("playing", _events_p93, pygame.K_p) == "paused", "按下暂停键后从playing变成paused"))
results_93_2.append(test_check(lambda: handle_pause_key("playing", _events_other93, pygame.K_p) == "playing", "按其他键不会触发暂停"))
results_93_2.append(test_check(lambda: handle_pause_key("paused", _events_p93, pygame.K_p) == "playing", "暂停时再按暂停键恢复游戏"))
print(f"\n通过 {sum(results_93_2)}/{len(results_93_2)} 项")


## Day 94：音效完善——给关键事件配上声音反馈

**学习目标**：掌握怎么给开始/得分/结束等关键事件分别配上对应的音效，并且在没有音效文件时也不会导致程序崩溃

### 📖 知识点讲解

**讲解要点**

- **一个正式一点的游戏，几乎每一个重要时刻都应该有声音反馈**：开始游戏有提示音、接到掉落物有"叮"的一声、游戏结束有另一种音效。可以用一个字典把"事件名字"（比如`"score"`、`"start"`、`"game_over"`）对应到具体的音效对象，需要用哪个音效时就按名字去查。
  **实际场景**：很多App的消息提示音、支付成功提示音，都是按"事件类型"分别配置的——不同的操作结果对应不同的声音。

- **写音效相关的代码时，特别要考虑"万一这个音效不存在怎么办"**——这时候程序不应该直接报错崩溃，而是"安静地什么都不做"。这种"遇到意外情况时选择优雅地跳过，而不是让整个程序崩溃"的写法，是这周"健壮性"主题的一个具体例子。
  **实际场景**：手机开了"静音模式"的时候，App该弹的通知、该做的操作还是照常执行，只是不发出声音——"声音播放失败或被禁用"不应该影响程序其他部分的正常运行。


### ✏️ 练习1：`play_sound_if_available(sound, enabled=True)`

接收一个音效对象`sound`（也可能是`None`，表示没有这个音效）和一个开关`enabled`（默认`True`）。如果`sound`不是`None`**并且**`enabled`是`True`，就调用`sound.play()`播放它，然后`return True`；否则什么都不做，`return False`。


In [ ]:
# ✏️ 你的代码
def play_sound_if_available(sound, enabled=True):
    pass


In [ ]:
# 🔍 检测
results_94_1 = []
try:
    _buf94 = struct.pack("<" + "h" * 100, *([0] * 100))
    _snd94 = pygame.mixer.Sound(buffer=_buf94)
    results_94_1.append(test_check(lambda: play_sound_if_available(_snd94) == True, "有音效且启用时播放并返回True"))
    results_94_1.append(test_check(lambda: play_sound_if_available(None) == False, "没有音效时不报错并返回False"))
    results_94_1.append(test_check(lambda: play_sound_if_available(_snd94, enabled=False) == False, "音效被禁用时不播放"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_94_1.extend([False, False, False])
print(f"\n通过 {sum(results_94_1)}/{len(results_94_1)} 项")


### ✏️ 挑战：`get_sound_for_event(event_name, sound_map)`

接收事件名字`event_name`（比如`"score"`）和一个"事件名→音效对象"的字典`sound_map`。用`.get(key, 默认值)`从`sound_map`里找`event_name`对应的音效，找不到就返回`None`（不要报错），`return`结果。


In [ ]:
# ✏️ 你的代码
def get_sound_for_event(event_name, sound_map):
    pass


In [ ]:
# 🔍 检测
results_94_2 = []
_sound_map94 = {"score": _snd94, "start": _snd94}
results_94_2.append(test_check(lambda: get_sound_for_event("score", _sound_map94) is _snd94, "根据事件名正确找到对应音效"))
results_94_2.append(test_check(lambda: get_sound_for_event("explosion", _sound_map94) is None, "找不到对应音效时返回None，不报错"))
print(f"\n通过 {sum(results_94_2)}/{len(results_94_2)} 项")


## Day 95：健壮性与边界情况——让程序不容易被意外输入搞崩溃

**学习目标**：巩固给已有函数加上防御性检查的习惯，并学会写一个配置校验函数，提前发现配置错误

### 📖 知识点讲解

**讲解要点**

- **到了收尾阶段，很值得回头检查一遍："如果传进来的参数是一些奇怪的值，我的函数会不会直接报错崩溃？"**——比如要调整音量的那个对象其实是`None`（音效还没加载好），如果不提前判断就直接调用`.set_volume(...)`，程序就会崩溃。给关键函数加上"先判断一下这个东西是否存在/是否合理，再继续往下走"的保护，是让程序变得"健壮"的核心思路。
  **实际场景**：很多正式软件在读取用户上传的文件之前，都会先检查"这个文件是不是空的、格式对不对"，而不是直接假设文件内容一定是对的。

- **一个游戏往往需要很多配置项，如果某一项配置不小心漏填了，最好能在程序刚启动的时候就发现，而不是等运行到那一行代码才突然报错**。可以写一个专门的"配置检查"函数，提前看看"所有必须有的配置项是不是都齐全了"，缺了就明确地说出"缺了哪几项"。
  **实际场景**：很多软件安装或启动时会先做一次"环境检查"，提前把问题列出来，而不是让用户在使用过程中随机碰到各种莫名其妙的报错。


### ✏️ 练习1：`safe_set_volume(sound, volume)`

接收音效对象`sound`（可能是`None`）和想设置的音量`volume`。如果`sound is None`，直接`return None`（不报错）。否则把`volume`限制在`0.0`到`1.0`之间（复用第12周学过的`max`/`min`写法），用`sound.set_volume(...)`设置音量，最后`return sound.get_volume()`。


In [ ]:
# ✏️ 你的代码
def safe_set_volume(sound, volume):
    pass


In [ ]:
# 🔍 检测
results_95_1 = []
try:
    _snd95 = pygame.mixer.Sound(buffer=_buf94)
    results_95_1.append(test_check(lambda: safe_set_volume(_snd95, 0.5) == 0.5, "正常音量设置成功"))
    results_95_1.append(test_check(lambda: safe_set_volume(_snd95, 1.5) == 1.0, "超过1的音量被限制在1.0"))
    results_95_1.append(test_check(lambda: safe_set_volume(None, 0.5) is None, "音效为None时不报错，返回None"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_95_1.extend([False, False, False])
print(f"\n通过 {sum(results_95_1)}/{len(results_95_1)} 项")


### ✏️ 挑战：`validate_game_config(config, required_keys)`

接收一个配置字典`config`和一份"必须包含哪些键"的列表`required_keys`。找出`required_keys`里有哪些键不在`config`里（可以用列表推导式：`[k for k in required_keys if k not in config]`），把这份"缺失的键"列表叫做`missing`。最后`return (len(missing) == 0, missing)`——一个元组，第一项表示"配置是否完整"，第二项是缺失的键列表（完整的话就是空列表）。


In [ ]:
# ✏️ 你的代码
def validate_game_config(config, required_keys):
    pass


In [ ]:
# 🔍 检测
results_95_2 = []
results_95_2.append(test_check(lambda: validate_game_config({"screen_w": 300, "screen_h": 200}, ["screen_w", "screen_h"]) == (True, []), "配置完整时验证通过"))
results_95_2.append(test_check(lambda: validate_game_config({"screen_w": 300}, ["screen_w", "screen_h", "max_misses"]) == (False, ["screen_h", "max_misses"]), "配置缺失时能正确列出缺失的键"))
print(f"\n通过 {sum(results_95_2)}/{len(results_95_2)} 项")


## Day 96：最终整合——把关卡、暂停、音效钩子都装进游戏类里

**学习目标**：练习把本周新增的几个功能（关卡表、暂停切换、音效钩子）整合进第13周已经搭好的游戏框架里

### 📖 知识点讲解

**讲解要点**

- **这一天不学新知识点，主要是"组装"**：把Day92的关卡表查询、Day93的暂停切换、Day94的音效钩子，一起放进上周已经写好的"完整游戏类"里，让这个类支持"按关卡表调整难度""可以暂停""关键事件会触发音效"这几项新能力。
  **实际场景**：软件工程里经常把一个个已经各自测试通过的小功能模块，逐步整合进主系统——这个"整合"阶段本身就是很重要的工作，经常会在各模块组合在一起时才暴露出新的小问题，所以整合之后再测试一遍是必须的。

- **整合的时候要特别注意"状态之间会不会打架"**：比如暂停状态下，核心玩法的"逐帧更新"（移动、掉落物下落、碰撞检测）应该被完全冻住，不能让掉落物在暂停的时候还偷偷往下掉。写`step()`、`move_player()`这类函数时，记得先检查"当前到底是不是可以执行这个操作的状态"。
  **实际场景**：很多应用在"后台"或"锁屏"状态下，会主动暂停一些本来在跑的任务（比如下载、计时器），避免资源浪费或出现奇怪的数据不一致——这跟游戏暂停时冻住玩法逻辑是同一个道理。


### ✏️ 挑战：`make_release_game(screen_w, screen_h, player_w, player_h, item_size, level_table, points_each, start_button_rect, restart_button_rect, pause_key, sounds=None, player_speed=15, max_misses=3)`

在函数内部定义一个表示"最终发布版游戏"的类，至少包含：

1. **初始数据**：`self.state = "menu"`；玩家矩形在窗口底部中间；`self.score = 0`、`self.misses = 0`、`self.level = 1`；`self.speed`用Day92的`get_level_config(1, level_table)["speed"]`算出来；`self.items`是空的精灵组；`self.sounds`如果传入的是`None`就用空字典`{}`代替。
2. **`handle_events(self, events)`方法**：`menu`状态下如果点到了开始按钮，切到`playing`，并用Day94的函数尝试播放`"start"`音效；`playing`或`paused`状态下，用Day93的`handle_pause_key`检查有没有按暂停键，更新`self.state`；`game_over`状态下如果点到了重新开始按钮，重置所有数据（分数/miss/关卡/速度都恢复到第1关的初始值，掉落物清空）并切回`playing`。
3. **`move_player(self, pressed_keys)`**：**只有`playing`状态才移动**，逻辑跟第10-13周一样（方向键+边界检测）。
4. **`spawn_item(self, x=None)`**：生成一个用当前`self.speed`下落的掉落物，加入`self.items`。
5. **`step(self)`**：**只有`playing`状态才处理**逐帧更新——更新掉落物位置，撞到玩家就加分、尝试播放`"score"`音效，并用Day92的`check_level_complete`检查这一关是否通关，通关就把`self.level`加1，`self.speed`更新成新关卡的速度；没撞到但掉出屏幕就算miss；miss次数够多就切到`game_over`并尝试播放`"game_over"`音效。

函数最后创建并`return`一个这个类的对象。


In [ ]:
# ✏️ 你的代码
def make_release_game(screen_w, screen_h, player_w, player_h, item_size, level_table, points_each,
                       start_button_rect, restart_button_rect, pause_key, sounds=None,
                       player_speed=15, max_misses=3):
    pass


In [ ]:
# 🔍 检测
results_96 = []

try:
    _start_rect96 = pygame.Rect(100, 80, 100, 40)
    _restart_rect96 = pygame.Rect(100, 100, 100, 40)
    _game96 = make_release_game(300, 200, 40, 10, 20, _table92, 10, _start_rect96, _restart_rect96, pygame.K_p,
                                 sounds={"score": _snd94, "start": _snd94, "game_over": _snd94},
                                 player_speed=15, max_misses=3)
    results_96.append(test_check(lambda: _game96.state == "menu" and _game96.speed == 20, "新游戏从menu状态开始，初始速度等于第一关速度"))
except Exception as e:
    print(f"❌ 初始状态测试运行出错：{type(e).__name__}: {e}")
    results_96.append(False)

try:
    _click_start96 = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 90), button=1)]
    _game96.handle_events(_click_start96)
    results_96.append(test_check(lambda: _game96.state == "playing", "点击开始按钮后进入playing状态"))
except Exception as e:
    print(f"❌ 开始按钮测试运行出错：{type(e).__name__}: {e}")
    results_96.append(False)

try:
    _game96.handle_events(_events_p93)
    results_96.append(test_check(lambda: _game96.state == "paused", "按暂停键后进入paused状态"))

    _before_pause_rect96 = _game96.player_rect.copy()
    _game96.move_player({pygame.K_RIGHT})
    results_96.append(test_check(lambda: _game96.player_rect == _before_pause_rect96, "暂停状态下玩家不能移动"))

    _game96.handle_events(_events_p93)
    results_96.append(test_check(lambda: _game96.state == "playing", "再按一次暂停键恢复游戏"))
except Exception as e:
    print(f"❌ 暂停功能测试运行出错：{type(e).__name__}: {e}")
    results_96.extend([False, False, False])

print()
print(f"通过 {sum(results_96)}/{len(results_96)} 项")


## Day 97：代码整理与游戏说明书

**学习目标**：练习写一个状态汇总函数，并据此为自己的游戏整理一份简单的说明文档

### 📖 知识点讲解

**讲解要点**

- **一份好的说明书，往往需要一些"摘要信息"来说明"现在游戏是什么情况"**——比如当前在第几关、分数多少、miss了几次。可以写一个小函数，把游戏对象身上分散的几个属性，统一打包成一份"状态摘要"（字典的形式），方便以后无论是显示在屏幕上、打印出来调试，还是写进说明书里，都能复用这同一份摘要逻辑。
  **实际场景**：很多后台管理系统都有一个"系统状态总览"页面，把分散在各处的数据汇总成一份摘要——这跟给游戏写一个"状态摘要"函数是同一个思路。

- **"游戏说明书"不需要很正式，但至少应该包含两部分内容：怎么玩（操作方式、规则），以及这个游戏里用到了哪些Python/pygame知识点**。建议找一点时间自己动手写一份，这也是对这100天学习成果的一次梳理和总结。
  **实际场景**：几乎所有正式发布的软件、游戏都会配一份"使用说明"或"README"，这是让别人能快速理解这个作品的重要部分，也是很多专业程序员的习惯。


### ✏️ 练习1：`get_game_summary(game)`

接收一个游戏对象`game`（它有`state`、`level`、`score`、`misses`这几个属性）。`return`一个字典：`{"state": game.state, "level": game.level, "score": game.score, "misses": game.misses}`。


In [ ]:
# ✏️ 你的代码
def get_game_summary(game):
    pass


In [ ]:
# 🔍 检测
results_97 = []
try:
    _summary96 = get_game_summary(_game96)
    results_97.append(test_check(lambda: _summary96 == {"state": "playing", "level": 1, "score": 0, "misses": 0}, "能正确汇总当前游戏状态，用于生成说明书信息"))
except Exception as e:
    print(f"❌ 运行出错：{type(e).__name__}: {e}")
    results_97.append(False)
print(f"\n通过 {sum(results_97)}/{len(results_97)} 项")


### ✏️ 动手任务（不自动检测）：写一份游戏说明书

找一点时间，给自己这100天做出来的游戏写一份简单的说明书（几段话就可以），至少包含：① 怎么玩（操作方式、规则、怎么赢/怎么算输） ② 这个游戏里用到了哪些Python/pygame知识点（比如字典、类、pygame事件、碰撞检测……列出来就好）。这份说明书留到Day98 Checkpoint之后的"发布会"上用。


## Day 98：第14周 Checkpoint（期末大项目最终完成版）

### 小测验（口头回答，答案由家长核对，这里不自动检测）

1. 为什么用"关卡表"（一份数据）来管理难度，比直接写很多个if/elif更灵活？如果玩的关卡超过了表里设计的最大关卡，比较稳妥的处理方式是什么？
2. "暂停"状态和"结束"状态有什么本质区别？为什么暂停之后还能从原来的进度继续，而不是从头开始？
3. 给音效相关的代码加保护（比如音效是`None`时不报错）属于这周讲的哪个主题？为什么这很重要？
4. 写一个"配置检查"函数，提前发现配置缺失，相比等程序运行到那一行才报错，好处是什么？
5. 整合多个功能到一个游戏类里时，为什么需要特别注意"暂停状态下，游戏玩法逻辑是不是真的被冻住了"？

### 挑战题：期末大项目——最终完成版

**背景**：这是整个100天计划编码内容的最后一次整合——把第92-97天做的"关卡表、暂停功能、音效钩子、健壮性检查、状态摘要"，跟第10-13周攒下的"核心玩法+完整主循环+开始/结束界面"全部拼在一起，做出一个真正可以拿去给别人演示的"最终完成版"。

**要求**：复用Day96写好的`make_release_game`，确认它满足：

1. **关卡驱动的难度**：分数达到关卡表设定的通关线时，关卡和速度正确提升；关卡用完之后依然能正常运行，不报错崩溃
2. **暂停功能**：`playing`状态下按暂停键能进入`paused`状态，再按一次能恢复；暂停期间，玩家移动和掉落物更新都被冻住
3. **音效钩子**：开始游戏、接到掉落物得分、游戏结束这几个关键时刻，都能触发对应的音效（没有配置音效时不报错）
4. **沿用上周的完整流程**：开始界面→核心玩法→结束界面→重新开始，重新开始要把所有数据（包括关卡、速度）都正确重置

**完成标准**
- [ ] 分数达到关卡表设定的通关线时，关卡和速度会正确提升；关卡用完后游戏依然能正常运行
- [ ] 暂停功能正常：暂停时玩法完全冻住，恢复后能从原来的进度继续
- [ ] 至少3个关键事件（开始/得分/结束）配置了音效钩子，且没有音效时不会报错
- [ ] 点击"重新开始"后，分数、miss次数、关卡、速度全部正确恢复到初始值
- [ ] 整个游戏能完整玩一整局，而且孩子能自己讲清楚这个版本比上周多了哪些功能


In [ ]:
# 🔍 检测（直接运行，检查Day96写的make_release_game是否满足期末最终版的所有要求）
results_98 = []

try:
    _table98 = [
        {"level": 1, "speed": 20, "score_to_pass": 20},
        {"level": 2, "speed": 30, "score_to_pass": 9999},
    ]
    _start_rect98 = pygame.Rect(100, 80, 100, 40)
    _restart_rect98 = pygame.Rect(100, 100, 100, 40)
    _game98 = make_release_game(300, 200, 40, 10, 20, _table98, 20, _start_rect98, _restart_rect98, pygame.K_p,
                                 sounds={}, player_speed=15, max_misses=3)
    _click_start98 = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 90), button=1)]
    _game98.handle_events(_click_start98)
    results_98.append(test_check(lambda: _game98.state == "playing", "游戏能正常从菜单进入playing状态"))
except Exception as e:
    print(f"❌ 初始流程测试运行出错：{type(e).__name__}: {e}")
    results_98.append(False)

try:
    _game98.spawn_item(x=130)
    for _ in range(9):
        _game98.step()
    results_98.append(test_check(lambda: _game98.score == 20 and _game98.level == 2 and _game98.speed == 30, "接住掉落物得分达到通关线后，正确升级并提速"))
except Exception as e:
    print(f"❌ 关卡升级测试运行出错：{type(e).__name__}: {e}")
    results_98.append(False)

try:
    _game98.spawn_item(x=0)
    _game98.spawn_item(x=280)
    _game98.spawn_item(x=50)
    for _ in range(11):
        _game98.step()
    results_98.append(test_check(lambda: _game98.misses == 3 and _game98.state == "game_over", "miss次数够多之后正确进入game_over状态"))
except Exception as e:
    print(f"❌ game_over测试运行出错：{type(e).__name__}: {e}")
    results_98.append(False)

try:
    _click_restart98 = [pygame.event.Event(pygame.MOUSEBUTTONDOWN, pos=(120, 110), button=1)]
    _game98.handle_events(_click_restart98)
    results_98.append(test_check(lambda: (_game98.state, _game98.score, _game98.misses, _game98.level, _game98.speed) == ("playing", 0, 0, 1, 20), "点击重新开始后，分数/关卡/速度/miss全部正确重置"))
except Exception as e:
    print(f"❌ 重新开始测试运行出错：{type(e).__name__}: {e}")
    results_98.append(False)

print()
passed = sum(results_98); total = len(results_98)
print(f"第14周Checkpoint：通过 {passed}/{total} 项")
if passed == total:
    print("🎉 恭喜完成第14周，也恭喜做出了期末大项目的最终完成版！接下来可以办一场小小的发布会：演示你的游戏，讲讲用到了哪些知识点，这100天的学习就圆满完成啦！")


---

恭喜完成第14周的所有练习，这也是100天Python学习计划编码部分的最后一周！这周把关卡表、暂停功能、音效钩子、健壮性检查全部整合进了上周的游戏框架里，做出了一个真正"正式"的作品。如果有检测没通过，可以回去看看对应的"讲解要点"和"实际场景"部分，想一想问题出在哪里，也强烈建议在自己电脑上用真实的窗口、鼠标和键盘，把这个最终版游戏从头玩到尾试一试，听一听音效、试一试暂停。接下来就是属于孩子自己的"发布会"时间——演示游戏、讲讲用到的知识点，为这100天画上一个圆满的句号！
